# ASR Performance Evaluation: Whisper-tiny on FLEURS (Azerbaijani)

This project evaluates the accuracy of Automatic Speech Recognition (ASR) for the Azerbaijani language using the OpenAI Whisper-tiny model against the Google FLEURS dataset.

To run this pipeline, you must install specific libraries for audio processing (librosa), model handling (transformers, datasets), and performance evaluation (evaluate, jiwer).

In [ ]:
!pip install -q "datasets<3.0.0" transformers evaluate jiwer librosa pandas torch


### Evaluation Pipeline and Metrics Calculation

This block contains the core logic for testing the model. It includes data ingestion, model configuration for the target language, and automated metric reporting.

In [ ]:
import torch
import pandas as pd
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import evaluate
import warnings
warnings.filterwarnings('ignore')

print("1. Google FLEURS (az) loading")
dataset = load_dataset("google/fleurs", "az_az", split="test[:200]", trust_remote_code=True)

print("2. Whisper-tiny model loading")
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)

forced_decoder_ids = processor.get_decoder_prompt_ids(language="az", task="transcribe")

print("3. Audio resampling")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print("4. Speech recognition in progress")
predictions = []
references = []

for i, item in enumerate(dataset):
    audio = item["audio"]
    input_features = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features.to(device)

    predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    predictions.append(transcription.strip())
    references.append(item["transcription"].strip())

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

wer_score = wer_metric.compute(predictions=predictions, references=references)
cer_score = cer_metric.compute(predictions=predictions, references=references)

print("\n" + "="*30)
print(f"Average WER: {wer_score * 100:.2f}%")
print(f"Average CER: {cer_score * 100:.2f}%")
print("="*30 + "\n")

results_df = pd.DataFrame({
    "Reference": references,
    "Prediction": predictions
})

def compute_row_wer(row):
    if not row['Reference'].strip(): return 1.0
    return wer_metric.compute(predictions=[row['Prediction']], references=[row['Reference']])

def compute_row_cer(row):
    if not row['Reference'].strip(): return 1.0
    return cer_metric.compute(predictions=[row['Prediction']], references=[row['Reference']])

results_df['WER'] = results_df.apply(compute_row_wer, axis=1)
results_df['CER'] = results_df.apply(compute_row_cer, axis=1)

results_df = results_df.sort_values(by="WER")

print("Ən yaxşı 5 nümunə:")
for idx, row in results_df.head(5).iterrows():
    print(f"Ref: {row['Reference']}\nPred: {row['Prediction']}\nWER: {row['WER']:.2f} | CER: {row['CER']:.2f}\n")

print("-" * 20)

print("Ən pis 5 nümunə:")
for idx, row in results_df.tail(5).iterrows():
    print(f"Ref: {row['Reference']}\nPred: {row['Prediction']}\nWER: {row['WER']:.2f} | CER: {row['CER']:.2f}\n")

results_df.to_csv("part_a_results.csv", index=False)
print("Results (with WER and CER) are saved in 'part_a_results.csv'")